# Detección de red: V4/p95 por estación y ataques aislados

Este experimento NO sustituye el modelo CCA de 22.6 ppb. Se entrenan hasta 33 modelos, cada uno con sus 32 vecinas, y se calibra su propio p95. El protocolo completo está en [docs/experimento_red_v4_p95.md](../docs/experimento_red_v4_p95.md).

**Qué cambia respecto a las gráficas anteriores:** se ataca cada mensaje por separado y se comprueba tanto su detección como el máximo diario contrafactual de la red. No se transfiere el recall de CCA a otras estaciones, no se usa un promedio de recalls para decidir sobre días y no se confunden los cruces del indicador de 155 ppb con contingencias oficiales.

**Periodo:** sólo el test cronológico común, después de sus primeras 24 horas de contexto. **Cobertura:** las estaciones sin entrenamiento/test quedan como no evaluadas, no como aciertos. Los demás mensajes observados siguen participando en el máximo diario.

**Corrección documentada:** el respaldo inicial del perfil usa sólo train, no el año completo. El resto conserva arquitectura, ventana y entrenamiento V4. Cada estación se guarda al terminar; una interrupción no pierde los modelos completados.


In [1]:
from pathlib import Path
import sys

# False por defecto: Run All no entrena accidentalmente.
# True autoriza entrenar SOLO modelos faltantes; los ya guardados se verifican y reutilizan.
ENTRENAR_FALTANTES = False

actual = Path.cwd().resolve()
RAIZ_RED = next(p for p in (actual, *actual.parents) if (p / 'src/ingest/rama.py').is_file())
if str(RAIZ_RED) not in sys.path:
    sys.path.insert(0, str(RAIZ_RED))
from src.detect.experimento_red import ejecutar_red
resumen_red = ejecutar_red(RAIZ_RED, entrenar_faltantes=ENTRENAR_FALTANTES)


estacion  n_train  n_test  evaluable                 motivo
     ACO     5630      59       True                       
     AJM     6049    1671       True                       
     AJU     2398       0      False sin ventanas de prueba
     ATI     5856    1681       True                       
     BJU     4326     478       True                       
     CAM     6222    1656       True                       
     CCA     6764    1684       True                       
     CHO     2984       0      False sin ventanas de prueba
     CUA     2784    1553       True                       
     CUT     6445    1604       True                       
     FAC     6518    1004       True                       
     FAR     6210       0      False sin ventanas de prueba
     GAM     5838     561       True                       
     HGM     6448    1671       True                       
     INN     6001    1562       True                       
     IZT     4638       0      False sin

Periodo comun: 2025-10-21 00:00:00 a 2025-12-31 23:00:00. 27/33 estaciones evaluables.


01/33 ACO: checkpoint verificado, NO reentrena.


02/33 AJM: checkpoint verificado, NO reentrena.


03/33 AJU: NO EVALUADA (sin ventanas de prueba)


...

Contrafactuales aislados: bit 4, 35704 mensajes observados.


Contrafactuales aislados: bit 5, 35704 mensajes observados.


Contrafactuales aislados: bit 6, 35704 mensajes observados.


Contrafactuales aislados: bit 7, 35704 mensajes observados.


 bit        tipo_dano  n_dias  dias_vulnerables_observados  dias_vulnerables_evaluables  dias_dano_no_detectado  dias_con_dano_sin_evaluacion  pct_dias_vulnerables_evaluables  pct_dias_dano_no_detectado  ataques_daninos_evaluables  ataques_daninos_detectados  deteccion_dano_pct
   1    anulada_fase1      72                            0                            0                       0                             0                         0.000000                    0.000000                           0                           0                 NaN
   1 cambio_banda_red      72                            0                            0                       0                             0                         0.000000                    0.000000                           0                           0                 NaN
   1      cruce_fase1      72                            0                            0                       0                             0                      

## Cómo leer los resultados

1. `resumen_modelos.csv`: p95, MAE y falsas alarmas sobre datos limpios por estación.
2. `metricas_pareadas.csv`: recall, precisión y F2 con ataques al 5 %; no confundir con el barrido exhaustivo.
3. `resumen_red_por_bit.csv`: días vulnerables y días con una oportunidad no detectada, contados directamente. `deteccion_dano_pct` es el recall condicionado a producir el daño indicado.
4. `eventos_aislados.csv.gz`: cada mensaje/bit, su predicción, lectura atacada y efecto en el máximo. La columna `evaluable` debe consultarse antes de interpretar `detectado`.

**El rojo cuenta días y el verde ataques dañinos.** Un día puede tener oportunidades tanto detectadas como no detectadas. La ausencia de oportunidades deja la detección indefinida. No se impone que las curvas crezcan o decrezcan. Las oportunidades sin modelo se reportan por separado, y los faltantes limitan las conclusiones sobre toda la red.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

CARPETA_RED = RAIZ_RED / 'results/multiestacion_v4_p95_2025'
for archivo in ['cobertura_estaciones.csv', 'resumen_modelos.csv', 'resumen_red_por_bit.csv']:
    print(archivo)
    with pd.option_context('display.max_columns', None, 'display.max_rows', None):
        display(pd.read_csv(CARPETA_RED / archivo).round(2))

# Presentacion: leyendas fuera del area de datos; no cambian modelos ni metricas.
tabla = pd.read_csv(CARPETA_RED / 'resumen_red_por_bit.csv')
for tipo in ['cruce_fase1', 'cambio_banda_red']:
    t = tabla[tabla.tipo_dano == tipo].sort_values('bit')
    for columna, nombre in [('pct_dias_dano_no_detectado', 'dano_no_detectado'),
                            ('pct_dias_vulnerables_evaluables', 'oportunidad_real')]:
        fig, ax = plt.subplots(figsize=(11, 7.5))
        ax.plot(t.bit, t[columna], 'o-', color='crimson', lw=2,
                label=('% de dias con >=1 ataque danino NO detectado' if nombre == 'dano_no_detectado'
                       else '% de dias con >=1 ataque danino evaluable'))
        ax.plot(t.bit, t.deteccion_dano_pct, 's--', color='seagreen', lw=2,
                label='% de ataques daninos evaluables detectados (p95 por estacion)')
        ax.set(ylim=(0, 105), xlabel='Bit atacado', ylabel='Porcentaje (%)')
        ax.set_xticks(range(1, 8), [f'{b}\n({2**b} ppb)' for b in range(1, 8)])
        ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=1))
        caso = ('Cruce del maximo diario observado — umbral 155 ppb' if tipo == 'cruce_fase1'
                else 'Cambio de banda del maximo diario observado')
        fig.suptitle(f'{nombre.replace("_", " ").capitalize()} por bit\n{caso}', fontsize=14, y=.98)
        ax.legend(loc='lower left', bbox_to_anchor=(0, 1.02), fontsize=9)
        for row in t.itertuples():
            ax.annotate(f'{getattr(row, columna):.1f}%', (row.bit, getattr(row, columna)),
                        xytext=(0, 8), textcoords='offset points', ha='center', color='crimson', fontsize=9)
            if pd.notna(row.deteccion_dano_pct):
                ax.annotate(f'{row.deteccion_dano_pct:.1f}%\n(n={row.ataques_daninos_evaluables})',
                            (row.bit, row.deteccion_dano_pct), xytext=(0, -29), textcoords='offset points',
                            ha='center', color='seagreen', fontsize=8)
            else:
                ax.text(row.bit, 96, 'Sin casos', ha='center', color='gray', fontsize=8)
        ax.grid(alpha=.25)
        fig.text(.08, .025, '72 dias de test. Rojo: dias / 72; verde: ataques daninos detectados / evaluables.\n'
                 'Cada ataque modifica un solo mensaje. n = casos daninos, no dias. Sin casos: deteccion indefinida.\n'
                 'Red observada: 27 objetivos evaluables de 33 candidatos; los faltantes no se consideran ceros.', fontsize=8)
        fig.subplots_adjust(top=.76, bottom=.20, left=.08, right=.98)
        fig.savefig(RAIZ_RED / 'docs/img/multiestacion_v4_p95_2025' / f'{nombre}_{tipo}.png', dpi=180)
        plt.show()
        plt.close(fig)


cobertura_estaciones.csv


,estacion,n_train,n_test,evaluable,motivo
0,ACO,5630,59,True,NaN
1,AJM,6049,1671,True,NaN
2,AJU,2398,0,False,sin ventanas de prueba
3,ATI,5856,1681,True,NaN
4,BJU,4326,478,True,NaN
5,CAM,6222,1656,True,NaN
6,CCA,6764,1684,True,NaN
7,CHO,2984,0,False,sin ventanas de prueba
8,CUA,2784,1553,True,NaN
9,CUT,6445,1604,True,NaN


resumen_modelos.csv


,estacion,umbral_ppb,n_train,n_test,n_features,epocas,mae_ppb,sesgo_ppb,fp_limpio,fpr_limpio_pct,segundos
0,ACO,13.77,5630,59,66,22,4.85,-2.22,4,6.78,12.02
1,AJM,22.16,6049,1671,66,14,12.06,-1.97,242,14.48,10.48
2,ATI,26.96,5856,1681,66,9,8.83,2.24,61,3.63,6.54
3,BJU,20.66,4326,478,66,18,12.28,9.66,80,16.74,8.57
4,CAM,17.37,6222,1656,66,18,7.93,-0.51,213,12.86,11.52
5,CCA,14.75,6764,1684,66,30,10.25,-1.96,360,21.38,18.70
6,CUA,29.44,2784,1553,66,11,11.13,1.83,73,4.70,4.70
7,CUT,15.63,6445,1604,66,22,8.15,3.26,227,14.15,13.69
8,FAC,16.85,6518,1004,66,17,9.71,1.70,177,17.63,11.37
9,GAM,19.23,5838,561,66,14,12.31,10.09,126,22.46,8.83


resumen_red_por_bit.csv


,bit,tipo_dano,n_dias,dias_vulnerables_observados,dias_vulnerables_evaluables,dias_dano_no_detectado,dias_con_dano_sin_evaluacion,pct_dias_vulnerables_evaluables,pct_dias_dano_no_detectado,ataques_daninos_evaluables,ataques_daninos_detectados,deteccion_dano_pct
0,1,anulada_fase1,72,0,0,0,0,0.00,0.00,0,0,NaN
1,1,cambio_banda_red,72,0,0,0,0,0.00,0.00,0,0,NaN
2,1,cruce_fase1,72,0,0,0,0,0.00,0.00,0,0,NaN
3,1,falsa_fase1,72,0,0,0,0,0.00,0.00,0,0,NaN
4,2,anulada_fase1,72,0,0,0,0,0.00,0.00,0,0,NaN
5,2,cambio_banda_red,72,9,9,4,0,12.50,5.56,10,6,60.00
6,2,cruce_fase1,72,1,1,0,0,1.39,0.00,1,1,100.00
7,2,falsa_fase1,72,1,1,0,0,1.39,0.00,1,1,100.00
8,3,anulada_fase1,72,0,0,0,0,0.00,0.00,0,0,NaN
9,3,cambio_banda_red,72,22,22,10,0,30.56,13.89,39,23,58.97


![dano_no_detectado_cruce_fase1](../docs/img/multiestacion_v4_p95_2025/dano_no_detectado_cruce_fase1.png)

![oportunidad_real_cruce_fase1](../docs/img/multiestacion_v4_p95_2025/oportunidad_real_cruce_fase1.png)

![dano_no_detectado_cambio_banda_red](../docs/img/multiestacion_v4_p95_2025/dano_no_detectado_cambio_banda_red.png)

![oportunidad_real_cambio_banda_red](../docs/img/multiestacion_v4_p95_2025/oportunidad_real_cambio_banda_red.png)

## Resultados y limites de esta corrida

Se entrenaron y guardaron **27 modelos**, uno por estacion evaluable. AJU, CHO, FAR, IZT, MON y TAH no tienen test; no se inventan resultados para ellas. El entrenamiento acumulado tardo aproximadamente **5 minutos**, con hasta 60 epocas y parada temprana.

Se evaluaron **249,928 escenarios aislados** (35,704 mensajes × 7 bits) durante **72 dias**, 21/oct–31/dic/2025. Todos los mensajes observados de ese periodo tienen detector; esto NO significa cobertura completa de la red: hubo 35,704 observaciones de 57,024 posiciones posibles. Los datos faltantes siguen siendo desconocidos.

### Cruces del maximo diario observado a 155 ppb

| Bit | Dias con falsa excedencia posible | Dias con alguna falsa excedencia NO detectada | Casos daninos detectados / evaluables |
|---:|---:|---:|---:|
| 1 | 0 | 0 | Sin casos |
| 2 | 1 | 0 | 1 / 1 |
| 3 | 0 | 0 | Sin casos |
| 4 | 2 | 1 | 5 / 6 |
| 5 | 9 | 0 | 46 / 46 |
| 6 | 9 | 0 | 46 / 46 |
| 7 | 72 | 0 | 14255 / 14255 |

**No hay anulaciones evaluables:** el maximo limpio del periodo fue 152 ppb. No estamos verificando proteccion de las cinco contingencias reales de otros meses ni simulando declaraciones oficiales. Un 100 % con un solo caso (bit 2) no acredita deteccion general de ese bit. La falta de monotonicidad entre bits 2 y 3 es posible: el efecto del flip depende del estado previo del bit, no solo de su peso.

### Cambio de banda del maximo diario

Los dias con alguna oportunidad no detectada fueron **0, 4, 10, 15, 16, 9 y 0**, para bits 1–7. El mayor conteo en este test es **16/72 = 22.22 % para bit 5**. Es un resultado distinto del cruce de 155 ppb: no mezclar ambos tipos de daño.

### Las falsas alarmas siguen siendo un problema

En test totalmente limpio, los modelos produjeron **5642/35704 = 15.80 %** de falsas alarmas en conjunto (suma de conteos, no promedio de porcentajes por estacion). El rango por estacion fue **3.63–26.64 %**. El p95 de entrenamiento no garantiza 5 % en test. No concluir mejora global solo por detectar ataques grandes.

Las tablas de precision/recall/F2 de los ataques pareados al 5 % se guardan en `metricas_pareadas.csv`, separadas del barrido exhaustivo. Un dia con al menos una oportunidad dañina no implica que haya ocurrido un ataque ni estima la frecuencia real de un adversario ciego.

**Verificacion:** Run All carga checkpoints sin entrenar; se comprobaron predicciones recargando CCA y ACO para bits 1, 5 y 7, y 100 maximos contrafactuales contra un recalculo literal de toda la red diaria. El modelo CCA anterior y su corte operativo de 22.6 ppb siguen intactos.


## Revisión del asesor — bits 0–7 y leyendas breves (29/sep/2026)

Las figuras anteriores quedan como historial. Aquí conservamos **Riesgo compuesto por bit**, invertimos detección a **no detección** y reunimos tres curvas: **Días vulnerables**, **No detección**, **Riesgo compuesto**. Las explicaciones se trasladan al pie. No se fuerza el cruce ni la posición del máximo.

**Las dos primeras figuras:** prueba multiestación de 72 días, 27 modelos guardados y p95 propio. Rojo = días con alguna falsa excedencia posible / 72; verde = ataques no detectados / todos los ataques del bit. Azul = producto de esos porcentajes / 100, un **indicador**, NO el conteo real de días con daño no detectado. Ese conteo se muestra por separado en la tabla. No confundir el recall global con el condicionado a ataques dañinos.

**La tercera figura:** 365 días de 2025, daño sin LSTM. Compara falsas activaciones con ocultamientos. Se sustituye un solo mensaje y se conserva el resto de la red diaria. Son cruces simulados de 155 ppb, no declaraciones oficiales ni pruebas de persistencia.

**Cambio respecto al notebook 08:** usamos flips reales y modelos por estación, no la franja ideal que supone sumar siempre ni el recall de CCA trasladado a toda la red. Por eso los números pueden cambiar. El bit 0 representa 1 ppb, el 7 representa 128 ppb. La evaluación del bit 0 reutiliza predicciones verificadas: no entrena ni recalibra.

![Riesgo compuesto revisado](../docs/img/revision_graficas_bits_0_7/riesgo_compuesto_red_bits_0_7.png)

![Días vulnerables y no detección](../docs/img/revision_graficas_bits_0_7/dias_vulnerables_no_deteccion_red_bits_0_7.png)

![Falsas activaciones y ocultamientos anuales](../docs/img/revision_graficas_bits_0_7/falsas_activaciones_ocultamientos_2025_bits_0_7.png)

[Protocolo y límites](../docs/revision_graficas_2026-09-29.md). La siguiente celda es independiente del kernel anterior: verifica las fuentes guardadas y reproduce **sólo esta revisión**, sin modificar figuras históricas.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual_revision = Path.cwd().resolve()
raiz_revision = next(p for p in (actual_revision, *actual_revision.parents)
                     if (p / 'src/ingest/rama.py').is_file())
if str(raiz_revision) not in sys.path:
    sys.path.insert(0, str(raiz_revision))
from experiments.graficas_revision.generar import generar

tabla_revision, tabla_anual_revision, figuras_revision = generar(raiz_revision)
print('TEST CON LSTM — ambas clases de daño, indicadores y conteos observados:')
with pd.option_context('display.max_columns', None, 'display.max_rows', None,
                       'display.float_format', '{:.3f}'.format):
    display(tabla_revision)
    print('AÑO COMPLETO SIN LSTM — falsas activaciones y ocultamientos:')
    display(tabla_anual_revision)
for archivo_revision in figuras_revision:
    print(archivo_revision.relative_to(raiz_revision))


### Cómo interpretar esta revisión

El indicador compuesto alcanza su máximo en bit 5 (1.756 %), seguido del bit 4 (1.650 %). **No demuestra que el mayor daño real no detectado esté en el bit 5**: en este test, los 46 ataques de bit 5 que provocan falsa excedencia fueron detectados. El único día con falsa excedencia posible no detectada aparece en bit 4 (1/72 = 1.389 %). La diferencia es precisamente por qué conservamos indicador y conteo real por separado.

El bit 0 tiene 5,640 detecciones de 35,704 ataques (15.797 %), pero no provoca falsas excedencias en este test. Su recall no equivale a esa misma proporción de ataques perjudiciales identificados. Las falsas alarmas limpias siguen siendo 15.80 %.

En el año completo, las falsas activaciones son posibles en **2, 4, 7, 9, 30, 63, 63 y 358 días** para bits 0–7; los ocultamientos, en **0, 0, 1, 1, 1, 2, 0 y 3 días**. Son oportunidades independientes, no ataques simultáneos ni sumas acumulativas. Puede haber más de una oportunidad en un mismo día, pero se cuenta una sola vez por bit y efecto.

Estos conteos no equivalen a modificar únicamente el valor máximo y olvidar las otras mediciones: otra estación u hora puede sostener la excedencia. Asimismo, un mensaje que no era máximo puede fabricar una. No atribuir detección del LSTM a los episodios de febrero–abril con un test de octubre–diciembre.
